# MOST availability figures

Editable notebook for paper figures based on `data/fla_cleaned.csv`. It filters to the paper analysis set: quantitative research articles with complete availability coding (N=10,480). The classification logic matches `stats_analysis.py` and `stats.html`:

- code availability: `yes` when `is_code_publicly_available` is true
- data availability: `available`, `cite`, `both`, or `none` from `is_data_repository_available` and `is_data_cited_or_linked`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "fla_cleaned.csv"
EXPECTED_ANALYSIS_N = 10480
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Palatino Linotype", "Book Antiqua", "Palatino", "Times New Roman"],
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#d8d1c8",
    "axes.labelcolor": "#2c3e50",
    "xtick.color": "#2c3e50",
    "ytick.color": "#2c3e50",
})

DATA_COLORS = {
    "available": "#2B7A9B",
    "cite": "#C98A3A",
    "both": "#4B8F55",
    "none": "#D8D1C8",
}
CODE_COLORS = {"yes": "#2B7A9B", "no": "#D8D1C8"}


In [ ]:
raw_df = pd.read_csv(DATA_PATH)

def boolish(series):
    return series.fillna(False).astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])

required_availability_fields = [
    "is_code_publicly_available",
    "is_data_cited_or_linked",
    "is_data_repository_available",
]
complete_availability = raw_df[required_availability_fields].notna().all(axis=1)
complete_availability &= raw_df[required_availability_fields].astype(str).apply(lambda col: col.str.strip().ne("")).all(axis=1)
df = raw_df[boolish(raw_df["is_quantitative_study"]) & complete_availability].copy()
assert len(df) == EXPECTED_ANALYSIS_N, f"Expected {EXPECTED_ANALYSIS_N:,} rows, found {len(df):,}"

df["code_availability"] = boolish(df["is_code_publicly_available"]).map({True: "yes", False: "no"})

data_cited = boolish(df["is_data_cited_or_linked"])
data_repo = boolish(df["is_data_repository_available"])
df["data_availability"] = "none"
df.loc[data_repo & ~data_cited, "data_availability"] = "available"
df.loc[data_cited & ~data_repo, "data_availability"] = "cite"
df.loc[data_repo & data_cited, "data_availability"] = "both"

df["topic"] = df["lda_topic"].fillna("Unknown").replace("", "Unknown")
df["primary_region"] = df["clean_primary_region"].fillna("Unknown").replace("", "Unknown")

df.shape, df[["code_availability", "data_availability"]].value_counts().head()


In [ ]:
def summarize(df, group_col):
    total = df.groupby(group_col).size().rename("n")
    code = pd.crosstab(df[group_col], df["code_availability"]).reindex(columns=["yes", "no"], fill_value=0)
    data = pd.crosstab(df[group_col], df["data_availability"]).reindex(columns=["available", "cite", "both", "none"], fill_value=0)
    out = pd.concat([total, code.add_prefix("code_"), data.add_prefix("data_")], axis=1).reset_index()
    out = out.rename(columns={group_col: "group"}).sort_values(["n", "group"], ascending=[False, True])
    for col in ["code_yes", "code_no", "data_available", "data_cite", "data_both", "data_none"]:
        out[f"{col}_pct"] = out[col] / out["n"] * 100
    return out

summaries = {
    "topic": summarize(df, "topic"),
    "journal": summarize(df, "journal"),
    "primary_region": summarize(df, "primary_region"),
}

summaries["topic"].head()


In [ ]:
def plot_stacked_percent(summary, value_cols, colors, title, filename, top_n=None, figsize=None):
    plot_df = summary.head(top_n).copy() if top_n else summary.copy()
    if "code_yes" in value_cols:
        sort_col = "code_yes_pct"
        ascending = False
    elif "data_none" in value_cols:
        sort_col = "data_none_pct"
        ascending = True
    else:
        sort_col = f"{value_cols[0]}_pct"
        ascending = False
    plot_df = plot_df.sort_values(sort_col, ascending=ascending).reset_index(drop=True)
    labels = plot_df["group"]
    pct_df = plot_df[value_cols].div(plot_df["n"], axis=0) * 100
    pct_df = pct_df.iloc[::-1]
    labels = labels.iloc[::-1]

    if figsize is None:
        figsize = (7.2, max(2.2, 0.32 * len(plot_df)))
    fig, ax = plt.subplots(figsize=figsize)
    left = pd.Series(0, index=pct_df.index, dtype=float)

    for col in value_cols:
        key = col.replace("code_", "").replace("data_", "")
        ax.barh(labels, pct_df[col], left=left, color=colors[key], label=key, height=0.74)
        left += pct_df[col]

    ax.set_xlim(0, 112)
    ax.set_xlabel("Share of articles (%)")
    ax.legend(ncol=len(value_cols), frameon=False, bbox_to_anchor=(0, 1.02), loc="lower left")
    ax.grid(axis="x", color="#eee8df", linewidth=0.8)
    ax.set_axisbelow(True)
    for y, row in enumerate(pct_df.itertuples(index=False)):
        left_edge = 0.0
        for col in value_cols:
            width = float(getattr(row, col))
            if width <= 0:
                continue
            if "data_none" in value_cols and col != "data_none":
                left_edge += width
                continue
            label = f"{width:.1f}%"
            if width >= 6:
                text_x = left_edge + width / 2
                text_color = "black" if col in {"code_no", "data_none"} else "white"
                ha = "center"
            else:
                text_x = left_edge + width + 0.5
                text_color = "#2c3e50"
                ha = "left"
            ax.text(text_x, y, label, va="center", ha=ha, fontsize=8, color=text_color)
            left_edge += width
    for y, n in enumerate(plot_df["n"][::-1]):
        ax.text(103.5, y, f"n={n:,}", va="center", ha="left", fontsize=9, color="#2c3e50")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{filename}.svg", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{filename}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{filename}.pdf", dpi=300, bbox_inches="tight")
    return fig, ax


In [ ]:
# Topic figures
plot_stacked_percent(
    summaries["topic"],
    ["code_yes", "code_no"],
    CODE_COLORS,
    "Public code availability by topic",
    "code_by_topic",
)
plot_stacked_percent(
    summaries["topic"],
    ["data_available", "data_cite", "data_both", "data_none"],
    DATA_COLORS,
    "Data availability by topic",
    "data_by_topic",
)


In [ ]:
# Optional: export editable summary tables from the notebook.
for key, summary in summaries.items():
    summary.to_csv(ROOT / "data" / f"notebook_stats_summary_by_{key}.csv", index=False)